# Sales Forecasting with PyTorch + W&B

A complete multi-stage ML pipeline with full reproducibility tracking.

**What this notebook does:**
1. **Ingests** raw retail sales data from UCI → logs as a W&B artifact
2. **Preprocesses** — cleans and aggregates to daily sales → logs as a processed artifact
3. **Trains** a PyTorch LSTM → logs checkpoints per epoch with aliases
4. **Registers** the best checkpoint with `staging` and `production` aliases

**Prerequisite:** run `wandb login` in your terminal first. It will prompt for your API key.

**What to look for at the end:** the W&B lineage graph showing every stage connected — raw data → processed data → model checkpoint.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import wandb
from sklearn.preprocessing import StandardScaler

In [ ]:
# ---- Project + data config ----
PROJECT = "sales-forecasting"
RAW_DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/"
    "Online%20Retail.xlsx"
)

# ---- Training config ----
TRAIN_CONFIG = {
    "learning_rate": 0.0005,
    "epochs": 15,
    "sequence_length": 30,
    "hidden_layer_size": 50,
    "seed": 1,
}

print(f"Project: {PROJECT}")
print(f"Training: {TRAIN_CONFIG}")

---

## Stage 1 — Data Ingestion

Download the raw retail sales dataset and log it as a W&B artifact.

**Why:** the raw data becomes a versioned, immutable snapshot. Every downstream stage references this exact version — no "which CSV did we use?" questions.

**Produces:** artifact `raw-sales-data`.

In [ ]:
def ingest_raw_data():
    """Download the raw retail sales dataset and log it to W&B as an artifact."""
    with wandb.init(project=PROJECT, job_type="ingest-data") as run:
        # Download
        print("Downloading raw data...")
        df = pd.read_excel(RAW_DATA_URL)
        filename = "raw_sales_data.csv"
        df.to_csv(filename, index=False)
        print(f"Saved {len(df):,} rows to {filename}")

        # Create artifact
        artifact = wandb.Artifact(
            name="raw-sales-data",
            type="raw_dataset",
            description="Raw online retail sales data from UCI.",
        )
        artifact.add_file(filename)
        run.log_artifact(artifact)
        print("Raw data artifact logged to W&B.")

In [ ]:
ingest_raw_data()

---

## Stage 2 — Data Preprocessing

Clean the raw data, aggregate to daily sales, and split into train / validation.

**What happens:**
- Drop rows with missing `CustomerID`, negative quantities, or zero prices
- Aggregate transaction-level data to a continuous daily time series
- Split chronologically: last 60 days are validation

**Why the split is chronological:** a random split on time series leaks future patterns into training.

**Consumes:** `raw-sales-data`.
**Produces:** `processed-sales-data` (train.csv + validation.csv).

In [ ]:
def preprocess_data():
    """Clean, aggregate, and split the raw sales data."""
    with wandb.init(project=PROJECT, job_type="preprocess-data") as run:
        # ---- Load raw artifact ----
        raw_artifact = run.use_artifact("raw-sales-data:latest")
        raw_dir = raw_artifact.download()
        df = pd.read_csv(f"{raw_dir}/raw_sales_data.csv")
        original_rows = len(df)

        # ---- Clean ----
        missing_customer = int(df["CustomerID"].isna().sum())
        df = df.dropna(subset=["CustomerID"])

        negative_qty = int((df["Quantity"] <= 0).sum())
        df = df[df["Quantity"] > 0]

        bad_price = int((df["UnitPrice"] <= 0).sum())
        df = df[df["UnitPrice"] > 0]

        # ---- Aggregate to daily ----
        df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
        df["Sales"] = df["Quantity"] * df["UnitPrice"]

        daily = (
            df.set_index("InvoiceDate")["Sales"]
            .resample("D").sum()
            .reset_index()
            .rename(columns={"InvoiceDate": "Date", "Sales": "TotalSales"})
        )
        daily = (
            daily.set_index("Date")
            .asfreq("D", fill_value=0)
            .reset_index()
        )

        # ---- Split chronologically ----
        train_df = daily[:-60]
        val_df = daily[-60:]

        train_df.to_csv("train.csv", index=False)
        val_df.to_csv("validation.csv", index=False)

        # ---- Log data quality metrics ----
        wandb.log({
            "original_row_count": original_rows,
            "cleaned_row_count": len(df),
            "rows_dropped_missing_customer": missing_customer,
            "rows_dropped_negative_quantity": negative_qty,
            "rows_dropped_bad_price": bad_price,
            "total_days": len(daily),
            "non_zero_days": int((daily["TotalSales"] > 0).sum()),
            "date_start": str(daily["Date"].min()),
            "date_end": str(daily["Date"].max()),
        })

        # ---- Log processed artifact ----
        artifact = wandb.Artifact(
            name="processed-sales-data",
            type="processed_dataset",
            description="Daily aggregated sales, split into train/validation.",
        )
        artifact.add_file("train.csv")
        artifact.add_file("validation.csv")
        run.log_artifact(artifact)
        print("Processed data artifact logged to W&B.")
        print(f"  Train days: {len(train_df):,}")
        print(f"  Val days:   {len(val_df):,}")

In [ ]:
preprocess_data()

---

## Stage 3 — Training

Train an LSTM to forecast daily sales.

**Steps:**
1. Load `processed-sales-data` from W&B
2. Normalize (scaler fit on training only — no leakage)
3. Build sliding-window datasets
4. Train the LSTM, logging `train_loss` and `val_loss` per epoch
5. On each new best validation loss, checkpoint the model as a new artifact version

**Checkpointing with aliases:** each checkpoint is logged with `best` and `epoch_N` aliases. The `best` alias always points at the current best — downstream code loads `sales-forecasting-lstm:best` without knowing the version number.

**Consumes:** `processed-sales-data`.
**Produces:** `sales-forecasting-lstm` (multiple versions, `best` alias).

In [ ]:
class SalesDataset(torch.utils.data.Dataset):
    """Turn a univariate time series into (window -> next value) pairs."""

    def __init__(self, data, sequence_length):
        self.data = torch.FloatTensor(data).view(-1)
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.data) - self.sequence_length - 1

    def __getitem__(self, index):
        return (
            self.data[index:index + self.sequence_length],
            self.data[index + self.sequence_length],
        )


class LSTMModel(nn.Module):
    """Single-layer LSTM for next-step forecasting."""

    def __init__(self, input_size=1, hidden_layer_size=100, output_size=1):
        super().__init__()
        self.hidden_layer_size = hidden_layer_size
        self.lstm = nn.LSTM(input_size, hidden_layer_size)
        self.linear = nn.Linear(hidden_layer_size, output_size)
        self.hidden_cell = (
            torch.zeros(1, 1, self.hidden_layer_size),
            torch.zeros(1, 1, self.hidden_layer_size),
        )

    def forward(self, input_seq):
        # input_seq: [batch, seq_len, 1] -> LSTM expects [seq_len, batch, input]
        lstm_out, self.hidden_cell = self.lstm(
            input_seq.permute(1, 0, 2),
            self.hidden_cell,
        )
        return self.linear(lstm_out[-1])


def set_seed(seed):
    """Fix random seeds across Python, NumPy, and PyTorch."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
def train_forecaster():
    """Train the LSTM and checkpoint to W&B."""
    set_seed(TRAIN_CONFIG["seed"])

    with wandb.init(
        project=PROJECT,
        job_type="training",
        config=TRAIN_CONFIG,
    ) as run:
        config = wandb.config

        # ---- Load processed artifact ----
        artifact = run.use_artifact("processed-sales-data:latest")
        artifact_dir = artifact.download()
        train_df = pd.read_csv(f"{artifact_dir}/train.csv")
        val_df = pd.read_csv(f"{artifact_dir}/validation.csv")

        train_series = pd.to_numeric(
            train_df["TotalSales"], errors="coerce"
        ).dropna().values
        val_series = pd.to_numeric(
            val_df["TotalSales"], errors="coerce"
        ).dropna().values

        # ---- Normalize (fit on train only) ----
        scaler = StandardScaler()
        train_scaled = scaler.fit_transform(train_series.reshape(-1, 1))
        val_scaled = scaler.transform(val_series.reshape(-1, 1))

        # ---- Datasets + loaders ----
        train_ds = SalesDataset(train_scaled, config.sequence_length)
        val_ds = SalesDataset(val_scaled, config.sequence_length)
        train_loader = torch.utils.data.DataLoader(
            train_ds, batch_size=1, shuffle=False
        )
        val_loader = torch.utils.data.DataLoader(
            val_ds, batch_size=1, shuffle=False
        )

        # ---- Model, loss, optimizer ----
        model = LSTMModel(hidden_layer_size=config.hidden_layer_size)
        loss_fn = nn.MSELoss()
        optimizer = torch.optim.Adam(
            model.parameters(), lr=config.learning_rate
        )

        # Track gradients + parameters in W&B
        wandb.watch(model, log_freq=100)

        # ---- Training loop ----
        best_loss = float("inf")

        for epoch in range(config.epochs):
            # Train
            model.train()
            train_losses = []
            for seq, label in train_loader:
                optimizer.zero_grad()
                # Reset hidden state per sequence — prevents leakage across windows
                model.hidden_cell = (
                    torch.zeros(1, 1, model.hidden_layer_size),
                    torch.zeros(1, 1, model.hidden_layer_size),
                )
                pred = model(seq.unsqueeze(-1))
                loss = loss_fn(pred, label.view(1, 1))
                loss.backward()
                optimizer.step()
                train_losses.append(loss.item())

            avg_train_loss = float(np.mean(train_losses))

            # Validate
            model.eval()
            val_losses = []
            with torch.no_grad():
                for seq, label in val_loader:
                    model.hidden_cell = (
                        torch.zeros(1, 1, model.hidden_layer_size),
                        torch.zeros(1, 1, model.hidden_layer_size),
                    )
                    pred = model(seq.unsqueeze(-1))
                    val_losses.append(loss_fn(pred, label).item())

            avg_val_loss = float(np.mean(val_losses))

            # Log per epoch
            wandb.log({
                "epoch": epoch,
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss,
            })
            print(
                f"Epoch {epoch:2d} | "
                f"train={avg_train_loss:.4f} | "
                f"val={avg_val_loss:.4f}"
            )

            # Checkpoint on improvement
            if avg_val_loss < best_loss:
                best_loss = avg_val_loss
                bundle = {
                    "model_state_dict": model.state_dict(),
                    "scaler": scaler,
                    "config": dict(config),
                }
                torch.save(bundle, "best_model_bundle.pth")

                artifact = wandb.Artifact(
                    name="sales-forecasting-lstm",
                    type="model",
                    metadata={"epoch": epoch, "val_loss": best_loss},
                )
                artifact.add_file("best_model_bundle.pth")
                run.log_artifact(
                    artifact,
                    aliases=["best", f"epoch_{epoch}"],
                )

        print(f"Training complete. Best val loss: {best_loss:.4f}")

In [ ]:
train_forecaster()

---

## Stage 4 — Model Registry

Register the best checkpoint in the W&B Registry and assign lifecycle aliases.

**Why the registry:** the checkpoint artifact is a raw pointer. The registry turns it into a **named, curated model** with a stable alias (`staging`, `production`) that downstream systems can load by name.

**Pattern:** load the `best` checkpoint, add `staging` and `production` aliases, and log its URI for reference.

**Consumes:** `sales-forecasting-lstm:best`.

In [ ]:
def register_and_promote():
    """
    Promote the best checkpoint with lifecycle aliases.

    W&B's programmatic registry linking is plan-dependent. This function
    adds `staging` and `production` aliases directly to the artifact —
    a portable pattern that works on all plans. If your W&B account has
    the Registry feature enabled, you can also link it via the UI.
    """
    with wandb.init(project=PROJECT, job_type="register") as run:
        artifact = run.use_artifact("sales-forecasting-lstm:best")

        print(f"Best checkpoint: {artifact.qualified_name}")
        print(f"Version:         {artifact.version}")
        print(f"Aliases:         {artifact.aliases}")
        print(f"Metadata:        {artifact.metadata}")

        # Add promotion aliases
        for alias in ("staging", "production"):
            if alias not in artifact.aliases:
                artifact.aliases.append(alias)
        artifact.save()

        print("\nAdded aliases: 'staging', 'production'")
        print(f"View run: {run.get_url()}")


register_and_promote()

---

## What to Look For in the W&B UI

Open the project URL printed above. You'll see:

**Runs tab** — four runs, one per pipeline stage:
- `ingest-data`
- `preprocess-data`
- `training`
- `register`

**Charts tab** — `train_loss` and `val_loss` per epoch, plus gradient histograms from `wandb.watch()`.

**Artifacts tab** — three artifacts:
- `raw-sales-data` (v0)
- `processed-sales-data` (v0)
- `sales-forecasting-lstm` (multiple versions, `best` alias)

**Lineage graph** (the payoff) — a visual chain:

```
raw-sales-data → [preprocess-data] → processed-sales-data → [training] → sales-forecasting-lstm:best
```

Every node is clickable. Click `sales-forecasting-lstm:best` → see the training run → see the processed data version → see the raw data version.

**That's the audit trail.** One click answers "how was this model created?"